In [21]:
import pandas as pd
import requests, zipfile, io, numpy as np
from scipy.spatial import cKDTree

GTFS_URL = "https://data.calgary.ca/download/npk7-z3bj/application%2Fx-zip-compressed"

response = requests.get(GTFS_URL)
z = zipfile.ZipFile(io.BytesIO(response.content))
trips = pd.read_csv(z.open('trips.txt'))
stops = pd.read_csv(z.open('stops.txt'))
stop_times = pd.read_csv(z.open('stop_times.txt'))

trip_to_stops = stop_times.groupby(stop_times['trip_id'].astype(str))['stop_id'].apply(list).to_dict()

def nearest_stop(lat, lon, trip_id, stops, trip_to_stoptimes):
    valid_stop_ids = trip_to_stoptimes.get(str(trip_id))
    if not valid_stop_ids:
        return None
    valid_stop_ids = [str(s) for s in valid_stop_ids]
    valid_stops = stops[stops['stop_id'].astype(str).isin(valid_stop_ids)]

    ref_lat_rad = np.radians(valid_stops['stop_lat'].mean())
    longitude_scale = np.cos(ref_lat_rad)

    if valid_stops.empty:
        return None
    stop_coords = valid_stops[['stop_lat', 'stop_lon']].copy()
    stop_coords['stop_lon'] *= longitude_scale
    stop_coords = stop_coords.values

    tree = cKDTree(stop_coords)
    distance, index = tree.query([lat, lon*longitude_scale])
    nearest_stop_id = valid_stops['stop_id'].iloc[index]
    print(f"nearest_stop_id={nearest_stop_id}, distance={distance}")
    return str(nearest_stop_id) if distance < 0.003 else None

test_trip_id = "74490923"
test_lat, test_lon = 51.127, -114.133
test_stop_id = "2955"

result = nearest_stop(test_lat, test_lon, test_trip_id, stops, trip_to_stops)
print("Matched stop:", result)
print("Is valid for this trip:", result in [str(s) for s in trip_to_stops.get(test_trip_id, [])])
print("Expected stop:", test_stop_id)
print(stops[stops['stop_id'].astype(str) == '2955'][['stop_lat', 'stop_lon']])

print("74490923" in trip_to_stops)


nearest_stop_id=2955, distance=0.021480569827930254
Matched stop: None
Is valid for this trip: False
Expected stop: 2955
     stop_lat    stop_lon
798  51.14845 -114.134825
True
